In [162]:
import os
import sys

try:
    # This will work if running as a .py script
    current_file_path = os.path.abspath(__file__)
    current_script_dir = os.path.dirname(current_file_path)
except NameError:
    # This will be executed if __file__ is not defined (e.g., in a Jupyter Notebook)
    current_script_dir = os.getcwd()

# Navigate up two levels to get to the 'GlobalLocal' directory
project_root = os.path.abspath(os.path.join(current_script_dir, '..', '..'))

# Add the 'GlobalLocal' directory to sys.path if it's not already there
if project_root not in sys.path:
    sys.path.insert(0, project_root)  # insert at the beginning to prioritize it

import pickle
import pandas as pd
import numpy as np

from src.analysis.decoding.decoding import plot_accuracies_with_multiple_sig_clusters
from src.analysis.config.condition_registry import CONDITION_REGISTRY

# imports for bar plots and ANOVA
import matplotlib.pyplot as plt
from scipy import stats
import pingouin as pg
from matplotlib.legend_handler import HandlerTuple
from matplotlib import lines as mlines


load in the master results files

In [163]:
def load_master_results_file(base_path, epochs_root_file, master_results_file):
    with open(os.path.join(base_path, epochs_root_file, master_results_file), "rb") as f:
        master_results = pickle.load(f)
    return master_results

In [164]:
base_path = "/cwork/jz421/BIDS-1.1_GlobalLocal/BIDS/derivatives/decoding/figs"
save_dir = "/hpc/home/jz421/coganlab/jz421/GlobalLocal/dcc_scripts/decoding/block_by_block_cns26_poster_plots"
epochs_root_file = "Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit"

# update these once done running decoding
results_files = {
    'congruency_blockA_results_file': "20260227_133015_MASTER_RESULTS_19_subs_all_elecs_LDA_20boots_5splits_5reps_repeat_unit_ev_0.9_stimulus_congruency_blockA_conditions.pkl",
    'congruency_blockB_results_file': "20260227_132957_MASTER_RESULTS_19_subs_all_elecs_LDA_20boots_5splits_5reps_repeat_unit_ev_0.9_stimulus_congruency_blockB_conditions.pkl",
    'congruency_blockC_results_file': "20260227_132958_MASTER_RESULTS_19_subs_all_elecs_LDA_20boots_5splits_5reps_repeat_unit_ev_0.9_stimulus_congruency_blockC_conditions.pkl",
    'congruency_blockD_results_file': "20260227_133008_MASTER_RESULTS_19_subs_all_elecs_LDA_20boots_5splits_5reps_repeat_unit_ev_0.9_stimulus_congruency_blockD_conditions.pkl",
    'switchType_blockA_results_file': "20260227_133008_MASTER_RESULTS_19_subs_all_elecs_LDA_20boots_5splits_5reps_repeat_unit_ev_0.9_stimulus_switchType_blockA_conditions.pkl",      
    'switchType_blockB_results_file': "20260227_133008_MASTER_RESULTS_19_subs_all_elecs_LDA_20boots_5splits_5reps_repeat_unit_ev_0.9_stimulus_switchType_blockB_conditions.pkl",
    'switchType_blockC_results_file': "20260227_140003_MASTER_RESULTS_19_subs_all_elecs_LDA_20boots_5splits_5reps_repeat_unit_ev_0.9_stimulus_switchType_blockC_conditions.pkl",
    'switchType_blockD_results_file': "20260227_141658_MASTER_RESULTS_19_subs_all_elecs_LDA_20boots_5splits_5reps_repeat_unit_ev_0.9_stimulus_switchType_blockD_conditions.pkl"
}

results = {}

for results_name, results_filepath in results_files.items():
    results[results_name] = load_master_results_file(base_path, epochs_root_file, results_filepath)


In [165]:
# Import the original NATURE_STYLE (you'll need to adjust this import based on your setup)
NATURE_STYLE = {
    'figure.figsize': (89/25.4, 89/25.4),
    'font.size': 12,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'axes.linewidth': 0.5,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.major.width': 0.5,
    'ytick.major.width': 0.5,
    'xtick.major.size': 2,
    'ytick.major.size': 2,
    'lines.linewidth': 1,
    'lines.markersize': 3,
    'legend.frameon': False,
    'legend.columnspacing': 0.5,
    'legend.handlelength': 1,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05
}

POSTER_STYLE = {
    **NATURE_STYLE,
    'lines.linewidth': 2,
    'font.size': 14,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 10,
}
    

In [166]:
def plot_block_decoding_comparison_dual_color(
    results_dict,
    block_metadata,
    dual_color_metadata,
    decoding_type,
    roi_to_plot='lpfc',
    save_dir='.',
    epochs_root_file='',
    ylim=(0.3, 0.9),
    single_column=True,
    show_legend=True,
    filename_suffix='dual_color',
):
    """
    Same data extraction as plot_block_decoding_comparison,
    but uses dual-color dashed lines instead of calling
    plot_accuracies_with_multiple_sig_clusters.
    """
    # --- Data extraction (copied from plot_block_decoding_comparison) ---
    first_key = next(iter(results_dict))
    first_results = results_dict[first_key]
    unit = first_results['metadata']['args']['unit_of_analysis']
    time_points = np.array(first_results['metadata']['time_window_centers'])
    
    save_directory = os.path.join(save_dir, epochs_root_file, f"{roi_to_plot}_poster_plots")
    os.makedirs(save_directory, exist_ok=True)
    
    accuracies_dict = {}
    shuffle_accs_list = []
    
    for block_key, meta in block_metadata.items():
        label = meta['label']
        block_results = results_dict[block_key]
        comp_key = next(iter(block_results['stats'].keys()))
        accuracies_dict[label] = block_results['stats'][comp_key][roi_to_plot][f'{unit}_true_accs']
        shuffle_accs_list.append(
            block_results['stats'][comp_key][roi_to_plot][f'{unit}_shuffle_accs']
        )
    
    pooled_chance = np.mean(np.stack(shuffle_accs_list, axis=0), axis=0)
    accuracies_dict['Chance'] = pooled_chance
    
    # --- Plotting ---
    fig, ax = plt.subplots(figsize=(89/25.4, 89/25.4 * 0.8))
    
    ylabel = {
        'congruency': 'Mean Congruency Decoding Accuracy',
        'switchType': 'Mean Switch Type Decoding Accuracy',
    }.get(decoding_type, 'Decoding Accuracy')
    
    legend_elements = []
    
    for label, accuracies in accuracies_dict.items():
        meta = dual_color_metadata[label]
        
        # Compute mean and SEM
        if accuracies.ndim == 2:
            mean_acc = np.mean(accuracies, axis=0)
            std_acc = np.std(accuracies, axis=0)
        else:
            mean_acc = accuracies
            std_acc = np.zeros_like(accuracies)
        
        if 'color_fg' in meta:
            # --- Two-layer trick ---
            ax.plot(time_points, mean_acc, '-',
                    color=meta['color_bg'], linewidth=2, zorder=3)
            ax.plot(time_points, mean_acc, '--',
                    color=meta['color_fg'], linewidth=2,
                    dashes=meta.get('dashes', (1, 1)), zorder=4)
            
            # # SEM shading - use bg color
            # if decoding_type == 'congruency':
            #     ax.fill_between(time_points,
            #                     mean_acc - std_acc, mean_acc + std_acc,
            #                     alpha=0.12, color=meta['color_fg'], linewidth=0)
            # elif decoding_type == 'switchType':
            #     ax.fill_between(time_points,
            #                     mean_acc - std_acc, mean_acc + std_acc,
            #                     alpha=0.12, color=meta['color_bg'], linewidth=0)
                
            # --- Proxy legend artist (stacked lines) ---
            from matplotlib import lines as mlines
            proxy_bg = mlines.Line2D([], [], linewidth=2, linestyle='-',
                                      color=meta['color_bg'])
            proxy_fg = mlines.Line2D([], [], linewidth=2, linestyle='--',
                                      dashes=(1, 1), color=meta['color_fg'])
            legend_elements.append(((proxy_bg, proxy_fg), label))
        else:
            # --- Single color (Chance) ---
            ax.plot(time_points, mean_acc,
                    color=meta['color_bg'],
                    linestyle=meta.get('linestyle', '--'),
                    linewidth=1, zorder=2)
            
            proxy = mlines.Line2D([], [], linewidth=1,
                                   linestyle=meta.get('linestyle', '--'),
                                   color=meta['color_bg'])
            legend_elements.append((proxy, label))
    
    # --- Axis formatting ---
    ax.axvline(x=0, color='#666666', linestyle='-', linewidth=0.5, alpha=0.5)
    ax.set_xlabel('Time from stimulus onset (s)', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_ylim(ylim)
    ax.set_xlim(-0.5, 1.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', labelsize=8)
    
    # --- Legend ---
    if show_legend:
        handles = [elem[0] for elem in legend_elements]
        labels = [elem[1] for elem in legend_elements]
        ax.legend(handles, labels, loc='upper right', fontsize=6,
                  frameon=False, handler_map={tuple: HandlerTuple(ndivide=None)})
        # HandlerTuple makes the stacked (bg, fg) tuples render overlaid
    
    plt.tight_layout()
    
    fname = f'{decoding_type}_block_comparison_{filename_suffix}.png'
    plt.savefig(os.path.join(save_directory, fname), dpi=600, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {os.path.join(save_directory, fname)}")

In [167]:
def plot_block_decoding_comparison(
    results_dict,
    block_metadata,
    decoding_type,
    roi_to_plot='lpfc',
    save_dir='.',
    epochs_root_file='',
    ylim=(0.3, 0.9),
    single_column=True,
    show_legend=True,
    filename_suffix='poster_style',
):
    first_key = next(iter(results_dict))
    first_results = results_dict[first_key]
    unit = first_results['metadata']['args']['unit_of_analysis']
    time_points = first_results['metadata']['time_window_centers']
    window_size = first_results['metadata']['args']['window_size']
    step_size = first_results['metadata']['args']['step_size']
    sampling_rate = first_results['metadata']['args']['sampling_rate']
    
    save_directory = os.path.join(save_dir, epochs_root_file, f"{roi_to_plot}_poster_plots")
    os.makedirs(save_directory, exist_ok=True)
    
    accuracies_dict = {}
    colors = {}
    linestyles = {}
    shuffle_accs_list = []
    
    for block_key, meta in block_metadata.items():
        label = meta['label']
        block_results = results_dict[block_key]
        comp_key = next(iter(block_results['stats'].keys()))
        
        accuracies_dict[label] = block_results['stats'][comp_key][roi_to_plot][f'{unit}_true_accs']
        colors[label] = meta['color']
        linestyles[label] = meta['linestyle']
        
        shuffle_accs_list.append(
            block_results['stats'][comp_key][roi_to_plot][f'{unit}_shuffle_accs']
        )
    
    pooled_chance = np.mean(np.stack(shuffle_accs_list, axis=0), axis=0)
    accuracies_dict['Chance'] = pooled_chance
    colors['Chance'] = '#949494'
    linestyles['Chance'] = '--'
    
    ylabel = {
        'congruency': 'Congruency Decoding Accuracy',
        'switchType': 'Switch Type Decoding Accuracy',
    }.get(decoding_type, 'Decoding Accuracy')
    
    plot_accuracies_with_multiple_sig_clusters(
        time_points=time_points,
        accuracies_dict=accuracies_dict,
        significance_clusters_dict={},
        colors=colors,
        linestyles=linestyles,
        ylabel=ylabel,
        ylim=ylim,
        show_chance_level=False,
        show_sig_legend=False,
        comparison_name=f'{decoding_type}_block_comparison',
        roi=roi_to_plot,
        save_dir=save_directory,
        filename_suffix=filename_suffix,
        window_size=window_size,
        step_size=step_size,
        sampling_rate=sampling_rate,
        single_column=single_column,
        show_legend=show_legend,
    )
    print(f"✅ Plot saved in: {save_directory}")

In [168]:
def compute_mean_accuracy_in_window(accuracies, time_window_centers, t_min=0.0, t_max=1.5):
    """
    Average decoding accuracy across time windows within [t_min, t_max].
    
    Parameters
    ----------
    accuracies : array-like, shape (n_repeats, n_time_windows)
        Decoding accuracies per bootstrap repeat per time window.
    time_window_centers : list or array, shape (n_time_windows,)
        Center of each time window in seconds (e.g., [-0.875, ..., 1.375]).
    t_min, t_max : float
        Window bounds in seconds for averaging.
    
    Returns
    -------
    mean_acc : array, shape (n_repeats,)
        Mean accuracy within the window for each repeat.
    """
    time_window_centers = np.array(time_window_centers)
    mask = (time_window_centers >= t_min) & (time_window_centers <= t_max)
    accuracies = np.array(accuracies)
    if accuracies.ndim == 1:
        return np.mean(accuracies[mask])
    else:
        return np.mean(accuracies[:, mask], axis=1)


def build_block_accuracy_df(results_dict, block_metadata, roi_to_plot='lpfc',
                             t_min=0.0, t_max=1.5):
    """
    Build a long-format DataFrame with columns:
        repeat, switch_proportion, congruency_proportion, block, poster_label, mean_accuracy
    
    Each row = one bootstrap repeat's mean accuracy across time windows in [t_min, t_max].
    
    Parameters
    ----------
    results_dict : dict
        Keys are block keys (e.g., 'congruency_blockA_results_file' or after renaming 'blockA').
        Values are master_results dicts with structure:
            results[block_key]['stats'][comp_key][roi]['repeat_true_accs'] -> (n_repeats, n_time_windows)
            results[block_key]['metadata']['time_window_centers'] -> list of floats
            results[block_key]['metadata']['args']['unit_of_analysis'] -> str
    block_metadata : dict
        Block definitions with 'switch_proportion', 'congruency_proportion', 
        'true_label', 'poster_label' keys.
    roi_to_plot : str
    t_min, t_max : float
        Time window range for averaging.
    """
    rows = []
    
    for block_key, meta in block_metadata.items():
        block_results = results_dict[block_key]
        
        # Time window centers from metadata (list of floats)
        time_window_centers = block_results['metadata']['time_window_centers']
        
        # Unit of analysis (should be 'repeat')
        unit = block_results['metadata']['args']['unit_of_analysis']
        
        # Get the comparison key (e.g., 'Stimulus_c_blockA_vs_Stimulus_i_blockA')
        comp_key = next(iter(block_results['stats'].keys()))
        
        # Shape: (n_repeats, n_time_windows), e.g. (100, 37)
        true_accs = block_results['stats'][comp_key][roi_to_plot][f'{unit}_true_accs']
        
        # Average across time windows in [t_min, t_max] -> (n_repeats,)
        mean_accs = compute_mean_accuracy_in_window(true_accs, time_window_centers, t_min, t_max)
        
        for repeat_idx, acc in enumerate(mean_accs):
            rows.append({
                'repeat': repeat_idx,
                'block': meta['true_label'],
                'poster_label': meta['poster_label'],
                'switch_proportion': meta['switch_proportion'],
                'congruency_proportion': meta['congruency_proportion'],
                'mean_accuracy': acc,
            })
    
    return pd.DataFrame(rows)


def run_2x2_anova(df):
    """
    Run a 2x2 repeated-measures ANOVA on mean_accuracy with factors:
        switch_proportion × congruency_proportion
    
    'repeat' is the subject/unit variable (bootstrap repeats).
    """
    df = df.copy()
    df['switch_prop_cat'] = df['switch_proportion'].map({0.25: 'low', 0.75: 'high'})
    df['cong_prop_cat'] = df['congruency_proportion'].map({0.25: 'low_cong', 0.75: 'high_cong'})
    
    aov = pg.rm_anova(
        data=df,
        dv='mean_accuracy',
        within=['switch_prop_cat', 'cong_prop_cat'],
        subject='repeat',
        detailed=True,
    )
    
    return aov


def run_block_ttests_vs_chance(df, chance=0.5):
    """
    One-sample t-tests for each block against chance level.
    Uses poster_label as the block identifier.
    """
    results = {}
    for poster_label in sorted(df['poster_label'].unique()):
        block_data = df[df['poster_label'] == poster_label]['mean_accuracy']
        if len(block_data) > 1:
            t_stat, p_value = stats.ttest_1samp(block_data, chance, alternative='greater')
        else:
            t_stat, p_value = np.nan, 1.0
        results[poster_label] = {
            't_stat': t_stat,
            'p_value': p_value,
            'mean': block_data.mean(),
            'sem': block_data.sem() if len(block_data) > 1 else np.nan,
            'n': len(block_data),
        }
    return results


def create_decoding_2x2_barplots(
    df,
    decoding_type='congruency',
    save_dir='.',
    bar_color='#4C72B0',
    chance_level=0.5,
    y_min=0.35,
    y_max=0.75,
    y_range_type='fixed',
    filename_suffix='',
):
    """
    Create a 2x2 bar plot for decoding accuracy by block type.
    
    Layout (poster labels):
    
                      25% Switch        75% Switch
    75% Inc (25% cong)   Poster D (true A)   Poster B (true B)
    25% Inc (75% cong)   Poster C (true C)   Poster A (true D)
    """
    os.makedirs(save_dir, exist_ok=True)
    
    # --- Run ANOVA ---
    anova_table = run_2x2_anova(df)
    print(f"\n{'='*60}")
    print(f"2x2 RM ANOVA: {decoding_type} decoding accuracy (over repeats)")
    print(f"{'='*60}")
    print(anova_table.to_string(index=False))
    print()
    
    # --- Run t-tests vs chance ---
    ttest_results = run_block_ttests_vs_chance(df, chance=chance_level)
    print(f"One-sample t-tests vs chance ({chance_level}):")
    for label, res in ttest_results.items():
        sig = '***' if res['p_value'] < .001 else '**' if res['p_value'] < .01 else '*' if res['p_value'] < .05 else 'ns'
        print(f"  {label}: M={res['mean']:.4f}, SEM={res['sem']:.4f}, "
              f"t({res['n']-1})={res['t_stat']:.3f}, p={res['p_value']:.4f} {sig}")
    
    # --- Summary stats per block ---
    summary = df.groupby(['switch_proportion', 'congruency_proportion', 'poster_label']).agg(
        mean=('mean_accuracy', 'mean'),
        sem=('mean_accuracy', 'sem'),
    ).reset_index()
    
    fig, axs = plt.subplots(2, 2, figsize=(8, 10), sharey=True)
    
    bar_width = 0.5
    bar_kw = {'edgecolor': 'black', 'linewidth': 1.5, 'capsize': 6}
    
    # Grid: (row, col) -> (switch_prop, cong_prop, poster_label)
    grid = {
        (0, 0): (0.25, 0.25, 'Poster D'),  # true A: 25S/75I
        (0, 1): (0.75, 0.25, 'Poster B'),  # true B: 75S/75I
        (1, 0): (0.25, 0.75, 'Poster C'),  # true C: 25S/25I
        (1, 1): (0.75, 0.75, 'Poster A'),  # true D: 75S/25I
    }
    
    if y_range_type == 'adaptive':
        min_v = (summary['mean'] - summary['sem']).min()
        max_v = (summary['mean'] + summary['sem']).max()
        padding = (max_v - min_v) * 0.4
        y_min = min(min_v - padding / 2, chance_level - 0.05)
        y_max = max_v + padding
    
    label_fs = 22
    tick_fs = 18
    title_fs = 20
    sig_fs = 22
    
    for (row, col), (sp, cp, poster_label) in grid.items():
        ax = axs[row, col]
        
        row_data = summary[(summary['switch_proportion'] == sp) & 
                           (summary['congruency_proportion'] == cp)]
        
        if len(row_data) == 0:
            continue
            
        mean_val = row_data['mean'].values[0]
        sem_val = row_data['sem'].values[0]
        
        ax.bar(0, mean_val, bar_width, yerr=sem_val, color=bar_color, **bar_kw)
        ax.axhline(y=chance_level, color='gray', linestyle='--', linewidth=1, alpha=0.7)
        
        ax.set_title(poster_label, fontsize=title_fs, fontweight='bold', pad=10)
        ax.set_ylim(y_min, y_max)
        ax.set_xlim(-0.8, 0.8)
        ax.set_xticks([])
        
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.spines['left'].set_linewidth(1.5)
        ax.tick_params(axis='y', labelsize=tick_fs, length=0)
        ax.grid(False)
        
        # Significance marker (vs chance)
        if poster_label in ttest_results:
            tres = ttest_results[poster_label]
            if tres['p_value'] < 0.05:
                sig_text = '***' if tres['p_value'] < .001 else '**' if tres['p_value'] < .01 else '*'
                ax.text(0, mean_val + sem_val + (y_max - y_min) * 0.03,
                        sig_text, ha='center', va='bottom', 
                        fontsize=sig_fs, fontweight='bold')
    
    # Only show y-axis labels on left column
    for row in range(2):
        axs[row, 1].tick_params(axis='y', labelleft=False)
    
    ylabel = 'Congruency Decoding Acc.' if decoding_type == 'congruency' else 'Switch Type Decoding Acc.'
    fig.text(0.02, 0.5, ylabel, va='center', rotation='vertical',
             fontsize=label_fs, fontweight='bold')
    
    fig.text(0.3, 0.96, '25% Switch', ha='center', fontsize=label_fs, fontweight='bold')
    fig.text(0.73, 0.96, '75% Switch', ha='center', fontsize=label_fs, fontweight='bold')
    
    fig.text(0.97, 0.72, '75% Inc', va='center', rotation=-90, fontsize=label_fs, fontweight='bold')
    fig.text(0.97, 0.30, '25% Inc', va='center', rotation=-90, fontsize=label_fs, fontweight='bold')
    
    # ANOVA annotation - leave this off for now
    anova_text_lines = []
    for _, row_data in anova_table.iterrows():
        source = row_data['Source']
        f_val = row_data.get('F', np.nan)
        p_val = row_data.get('p-unc', np.nan)
        np2 = row_data.get('np2', np.nan)
        sig = '***' if p_val < .001 else '**' if p_val < .01 else '*' if p_val < .05 else ''
        anova_text_lines.append(f"{source}: F={f_val:.2f}, p={p_val:.4f}{sig}, ηp²={np2:.3f}")
    
    # anova_text = '\n'.join(anova_text_lines)
    # fig.text(0.5, -0.02, anova_text, ha='center', va='top', fontsize=12,
    #          fontstyle='italic', family='monospace',
    #          bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.8))
    
    plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95])
    
    fname = f'{decoding_type}_decoding_2x2_barplot_{y_range_type}'
    if filename_suffix:
        fname += f'_{filename_suffix}'
    
    save_path = os.path.join(save_dir, f'{fname}.png')
    plt.savefig(save_path, dpi=600, bbox_inches='tight')
    print(f"\n✅ Saved: {save_path}")
    plt.close()
    
    return anova_table, ttest_results


# =============================================================================
# ALTERNATIVE: Grouped bar plot (all 4 blocks side by side — cleaner for poster)
# =============================================================================

def create_decoding_grouped_barplot(
    df,
    decoding_type='congruency',
    save_dir='.',
    chance_level=0.5,
    y_min=0.35,
    y_max=0.75,
    y_range_type='fixed',
    filename_suffix='poster',
):
    """
    All 4 blocks as side-by-side bars in a single panel, in POSTER order (A, B, C, D).
    Colored by switch proportion, hatched by inc proportion.
    """
    os.makedirs(save_dir, exist_ok=True)
    
    anova_table = run_2x2_anova(df)
    ttest_results = run_block_ttests_vs_chance(df, chance=chance_level)
    
    print(f"\n{'='*60}")
    print(f"2x2 RM ANOVA: {decoding_type} decoding accuracy (over repeats)")
    print(f"{'='*60}")
    print(anova_table.to_string(index=False))
    
    summary = df.groupby(['poster_label', 'switch_proportion', 'congruency_proportion']).agg(
        mean=('mean_accuracy', 'mean'),
        sem=('mean_accuracy', 'sem'),
    ).reset_index()
    
    # Poster order: A, B, C, D
    # Poster A = true D (75S/25I), Poster B = true B (75S/75I),
    # Poster C = true C (25S/25I), Poster D = true A (25S/75I)
    poster_order = [
        {'poster': 'Poster A', 'sp': 0.75, 'cp': 0.75, 'x_label': '75S/25I'},
        {'poster': 'Poster B', 'sp': 0.75, 'cp': 0.25, 'x_label': '75S/75I'},
        {'poster': 'Poster C', 'sp': 0.25, 'cp': 0.75, 'x_label': '25S/25I'},
        {'poster': 'Poster D', 'sp': 0.25, 'cp': 0.25, 'x_label': '25S/75I'},
    ]
    
    color_map = {0.25: '#5B9BD5', 0.75: '#ED7D31'}  # blue=25% switch, orange=75% switch
    hatch_map = {0.25: '/', 0.75: ''}  # hatch for 75% inc (low cong_prop = 0.25)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    x = np.arange(len(poster_order))
    bar_width = 0.6
    
    means, sems, colors_list, hatches_list, labels = [], [], [], [], []
    for b in poster_order:
        row = summary[(summary['switch_proportion'] == b['sp']) & 
                       (summary['congruency_proportion'] == b['cp'])]
        means.append(row['mean'].values[0])
        sems.append(row['sem'].values[0])
        colors_list.append(color_map[b['sp']])
        hatches_list.append(hatch_map[b['cp']])
        labels.append(b['x_label'])
    
    if y_range_type == 'adaptive':
        min_v = min(m - s for m, s in zip(means, sems))
        max_v = max(m + s for m, s in zip(means, sems))
        padding = (max_v - min_v) * 0.4
        y_min = min(min_v - padding / 2, chance_level - 0.03)
        y_max = max_v + padding
    
    bars = ax.bar(x, means, bar_width, yerr=sems, capsize=6,
                  color=colors_list, edgecolor='black', linewidth=1.5)
    
    for bar, hatch in zip(bars, hatches_list):
        bar.set_hatch(hatch)
    
    ax.axhline(y=chance_level, color='gray', linestyle='--', linewidth=1.2, alpha=0.7,
               label=f'Chance ({chance_level})')
    
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=16, fontweight='bold')
    ax.tick_params(axis='x', length=0)
    ax.tick_params(axis='y', labelsize=16, length=0)
    
    ylabel = 'Congruency Decoding Acc.' if decoding_type == 'congruency' else 'Switch Type Decoding Acc.'
    ax.set_ylabel(ylabel, fontsize=20, fontweight='bold', labelpad=10)
    ax.set_xlabel('Block Type (Switch% / Inc%)', fontsize=18, fontweight='bold', labelpad=10)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.grid(False)
    
    # Significance markers
    buffer = (y_max - y_min) * 0.02
    sig_fs = 20
    
    for i, b in enumerate(poster_order):
        poster_label = b['poster']
        if poster_label in ttest_results:
            tres = ttest_results[poster_label]
            if tres['p_value'] < 0.05:
                sig_text = '***' if tres['p_value'] < .001 else '**' if tres['p_value'] < .01 else '*'
                ax.text(x[i], means[i] + sems[i] + buffer, sig_text,
                        ha='center', va='bottom', fontsize=sig_fs, fontweight='bold')
    
    # ANOVA annotation - leave this off for now
    anova_parts = []
    for _, r in anova_table.iterrows():
        src = r['Source']
        p = r.get('p-unc', np.nan)
        sig = '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else 'ns'
        short_src = src.replace('switch_prop_cat', 'SP').replace('cong_prop_cat', 'CP').replace(' * ', '×')
        anova_parts.append(f"{short_src}: p={p:.3f}{sig}")
    
    # ax.text(0.02, 0.98, '\n'.join(anova_parts), transform=ax.transAxes,
    #         fontsize=11, va='top', fontstyle='italic',
    #         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    
    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=color_map[0.25], edgecolor='black', label='25% Switch'),
        Patch(facecolor=color_map[0.75], edgecolor='black', label='75% Switch'),
        Patch(facecolor='white', edgecolor='black', hatch='/', label='75% Inc'),
        Patch(facecolor='white', edgecolor='black', label='25% Inc'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=12, framealpha=0.8)
    
    plt.tight_layout()
    
    fname = f'{decoding_type}_decoding_grouped_barplot_{y_range_type}'
    if filename_suffix:
        fname += f'_{filename_suffix}'
    save_path = os.path.join(save_dir, f'{fname}.png')
    plt.savefig(save_path, dpi=600, bbox_inches='tight')
    print(f"✅ Saved: {save_path}")
    plt.close()
    
    return anova_table, ttest_results



In [169]:
block_metadata_anova = {
    'blockA': {'true_label': 'A', 'poster_label': 'Poster D',
               'switch_proportion': 0.25, 'congruency_proportion': 0.25},
    'blockB': {'true_label': 'B', 'poster_label': 'Poster B',
               'switch_proportion': 0.75, 'congruency_proportion': 0.25},
    'blockC': {'true_label': 'C', 'poster_label': 'Poster C',
               'switch_proportion': 0.25, 'congruency_proportion': 0.75},
    'blockD': {'true_label': 'D', 'poster_label': 'Poster A',
               'switch_proportion': 0.75, 'congruency_proportion': 0.75},
}

block_metadata_timecourse = {
    'blockD': {'label': 'Poster A (25I/75S)', 'color': '#FF7E79', 'linestyle': '--'},
    'blockB': {'label': 'Poster B (75I/75S)', 'color': '#DE8F05', 'linestyle': '--'},
    'blockC': {'label': 'Poster C (25I/25S)', 'color': '#FF7E79', 'linestyle': '-'},
    'blockA': {'label': 'Poster D (75I/25S)', 'color': '#DE8F05', 'linestyle': '-'},
}

dual_color_metadata = {
    'Poster A (25I/75S)': {'color_bg': '#0173B2', 'color_fg': '#FF7E79', 'dashes': (1, 1)},
    'Poster B (75I/75S)': {'color_bg': '#0173B2', 'color_fg': '#DE8F05', 'dashes': (1, 1)},
    'Poster C (25I/25S)': {'color_bg': '#05B0F0', 'color_fg': '#FF7E79', 'dashes': (1, 1)},
    'Poster D (75I/25S)': {'color_bg': '#05B0F0', 'color_fg': '#DE8F05', 'dashes': (1, 1)},
    'Chance':             {'color_bg': '#949494', 'linestyle': '--'},
}

In [170]:
# --- Shared key renaming ---
congruency_results = {
    k.replace('congruency_', '').replace('_results_file', ''): v
    for k, v in results.items() if 'congruency' in k
}
switch_results = {
    k.replace('switchType_', '').replace('_results_file', ''): v
    for k, v in results.items() if 'switchType' in k
}

save_directory = os.path.join(save_dir, epochs_root_file, "lpfc_poster_plots")

# --- 1. Time-course plots (use block_metadata_timecourse) ---
for dtype, res in [('congruency', congruency_results), ('switchType', switch_results)]:
    plot_block_decoding_comparison(
        results_dict=res,
        block_metadata=block_metadata_timecourse,
        decoding_type=dtype,
        roi_to_plot='lpfc',
        save_dir=save_dir,
        epochs_root_file=epochs_root_file,
        show_legend=True,
    )

# --- 2. ANOVA + bar plots (use block_metadata_anova) ---
for dtype, res in [('congruency', congruency_results), ('switchType', switch_results)]:
    df = build_block_accuracy_df(
        results_dict=res,
        block_metadata=block_metadata_anova,
        roi_to_plot='lpfc',
        t_min=0.0,
        t_max=1.5,
    )
    
    # Option A: 2x2 grid
    anova, ttests = create_decoding_2x2_barplots(
        df, decoding_type=dtype, save_dir=save_directory, y_range_type='adaptive',
    )
    
    # Option B: Grouped bars (poster-friendly)
    anova, ttests = create_decoding_grouped_barplot(
        df, decoding_type=dtype, save_dir=save_directory, y_range_type='adaptive',
    )

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Saved Nature-style plot with multiple significance clusters to: /hpc/home/jz421/coganlab/jz421/GlobalLocal/dcc_scripts/decoding/block_by_block_cns26_poster_plots/Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit/lpfc_poster_plots/congruency_block_comparison_lpfc_poster_style.pdf
✅ Plot saved in: /hpc/home/jz421/coganlab/jz421/GlobalLocal/dcc_scripts/decoding/block_by_block_cns26_poster_plots/Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit/lpfc_poster_plots
Saved Nature-style plot with multiple significance clusters to: /hpc/home/jz421/coganlab/jz421/GlobalLocal/dcc_scripts/decoding/block_by_block_cns26_poster_plots/Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLengt

In [171]:
for dtype, res in [('congruency', congruency_results), ('switchType', switch_results)]:
    plot_block_decoding_comparison_dual_color(
        results_dict=res,
        block_metadata=block_metadata_timecourse,  # same as before
        dual_color_metadata=dual_color_metadata,     # the new dict
        decoding_type=dtype,
        roi_to_plot='lpfc',
        save_dir=save_dir,
        epochs_root_file=epochs_root_file,
        show_legend=True,
    )

✅ Saved: /hpc/home/jz421/coganlab/jz421/GlobalLocal/dcc_scripts/decoding/block_by_block_cns26_poster_plots/Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit/lpfc_poster_plots/congruency_block_comparison_dual_color.png
✅ Saved: /hpc/home/jz421/coganlab/jz421/GlobalLocal/dcc_scripts/decoding/block_by_block_cns26_poster_plots/Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit/lpfc_poster_plots/switchType_block_comparison_dual_color.png
